# LAB 01 - Local Python

Notebook này chạy trên Google Colab bằng repository GitHub `NguyenDucThang-tb/NLP`. Cell đầu sẽ clone repository và tự tìm dataset.

Chạy các cell theo thứ tự từ trên xuống. Hãy hoàn thành `prediction.md` trước khi xem kết quả.

In [1]:
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/NguyenDucThang-tb/NLP.git'
REPO_DIR = Path('/content/NLP')
DATA_FILENAME = 'c4-train.00000-of-01024-30K.json'

if not (REPO_DIR / '.git').exists():
    if REPO_DIR.exists():
        raise RuntimeError(f'{REPO_DIR} tồn tại nhưng không phải Git repository. Hãy xóa thư mục này rồi chạy lại cell.')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

matches = list(REPO_DIR.rglob(DATA_FILENAME))
if not matches:
    raise FileNotFoundError(
        f'Không tìm thấy {DATA_FILENAME} trong repository. '
        'Hãy kiểm tra file đã được push lên GitHub chưa.'
    )
DATA_PATH = matches[0]
RESULTS_PATH = DATA_PATH.parent / 'part_D_results.csv'
print('Dataset:', DATA_PATH)
print('Dataset size (MB):', round(DATA_PATH.stat().st_size / 1024**2, 2))

Dataset: /content/NLP/Lab1/c4-train.00000-of-01024-30K.json
Dataset size (MB): 65.69


In [2]:
import csv
import json
import math
import re
from collections import Counter

with DATA_PATH.open(encoding='utf-8') as stream:
    documents = [json.loads(line) for line in stream]
texts = [item.get('text', '') for item in documents]
print(f'Number of documents: {len(documents):,}')
print('First document preview:', texts[0][:300])

Number of documents: 30,000
First document preview: Beginners BBQ Class Taking Place in Missoula!
Do you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join World Class BBQ Champion, Tony Balay from Lonestar Smoke Rangers. He will be teaching a beginner level class fo


In [3]:
TOKEN_RE = re.compile(r'(?u)\b\w+\b')
PUNCT_RE = re.compile(r'[^\w\s]', flags=re.UNICODE)

def normalized(text):
    return TOKEN_RE.findall(PUNCT_RE.sub(' ', text.lower()))

def extended(text):
    words = normalized(text)
    grams = [f'char3:{word[i:i+3]}' for word in words for i in range(max(0, len(word) - 2))]
    return words + grams

def fit_sparse_tfidf(raw_documents, tokenizer):
    tokenized = [tokenizer(text) for text in raw_documents]
    document_frequency = Counter()
    for tokens in tokenized:
        document_frequency.update(set(tokens))
    vocabulary = {term: index for index, term in enumerate(sorted(document_frequency))}
    number_of_documents = len(raw_documents)
    idf = {term: math.log(number_of_documents / df) for term, df in document_frequency.items()}
    vectors = []
    for tokens in tokenized:
        counts = Counter(tokens)
        total = len(tokens)
        vector = {vocabulary[term]: (count / total) * idf[term] for term, count in counts.items() if total and idf[term] != 0}
        vectors.append(vector)
    return tokenized, vocabulary, document_frequency, idf, vectors

def cosine_sparse(first, second):
    if len(first) > len(second):
        first, second = second, first
    dot_product = sum(value * second.get(index, 0.0) for index, value in first.items())
    first_norm = math.sqrt(sum(value * value for value in first.values()))
    second_norm = math.sqrt(sum(value * value for value in second.values()))
    return dot_product / (first_norm * second_norm) if first_norm and second_norm else 0.0

def search(query, model, tokenizer, top_k=5):
    if top_k <= 0:
        raise ValueError('top_k must be greater than zero')
    tokenized, vocabulary, document_frequency, idf, vectors = model
    query_tokens = tokenizer(query)
    counts = Counter(query_tokens)
    total = len(query_tokens)
    query_vector = {vocabulary[term]: (count / total) * idf[term] for term, count in counts.items() if term in vocabulary and total and idf[term] != 0}
    scores = [(index, cosine_sparse(query_vector, vector)) for index, vector in enumerate(vectors)]
    ranked = sorted(scores, key=lambda pair: (-pair[1], pair[0]))[:top_k]
    return [(rank, index, score) for rank, (index, score) in enumerate(ranked, start=1)]

## Part D - Retrieval experiment

In [ ]:
models = {}
for name, tokenizer in {'B_normalized': normalized, 'C_extended': extended}.items():
    model = fit_sparse_tfidf(texts, tokenizer)
    models[name] = model
    tokenized, vocabulary, document_frequency, idf, vectors = model
    non_zero = sum(len(set(tokens)) for tokens in tokenized)
    total_entries = len(tokenized) * len(vocabulary)
    sparsity = 1 - non_zero / total_entries
    average_tokens = sum(map(len, tokenized)) / len(tokenized)
    print(name, {'vocabulary': len(vocabulary), 'average_tokens': round(average_tokens, 2), 'sparsity': round(sparsity, 6)})

B_normalized {'vocabulary': 193837, 'average_tokens': 369.7, 'sparsity': 0.999122}


In [ ]:
queries = [
    'medical image classification',
    'transformer language model',
    'deep learning healthcare',
    'natural language processing',
    'computer vision medical diagnosis',
    'neural network training',
]

rows = []
for query in queries:
    print('\nQUERY:', query)
    results = search(query, models['B_normalized'], normalized, top_k=5)
    for rank, document_id, score in results:
        preview = texts[document_id][:240].replace('\n', ' ')
        print({'rank': rank, 'document_id': document_id, 'similarity': round(score, 6), 'preview': preview})
        rows.append([query, rank, document_id, f'{score:.8f}', preview])

with RESULTS_PATH.open('w', newline='', encoding='utf-8') as stream:
    writer = csv.writer(stream)
    writer.writerow(['query', 'rank', 'document_id', 'similarity', 'document_preview'])
    writer.writerows(rows)
print(f'Saved {len(rows)} rows to {RESULTS_PATH}')

## Part F - Evaluation và error analysis

Đọc document preview trong Part D rồi tự điền các document ID liên quan. Không dùng score đơn thuần để gắn nhãn.

In [ ]:
# TODO: tự điền document IDs relevant sau khi đọc nội dung document.
RELEVANT = {
    # 'medical image classification': {123, 456},
}

def evaluate_query(query, relevant, k=5):
    retrieved = [document_id for _, document_id, _ in search(query, models['B_normalized'], normalized, top_k=k)]
    relevant = set(relevant)
    hits = [document_id for document_id in retrieved if document_id in relevant]
    precision_at_k = len(hits) / k
    recall_at_k = len(hits) / len(relevant) if relevant else 0.0
    reciprocal_rank = next((1 / rank for rank, document_id in enumerate(retrieved, start=1) if document_id in relevant), 0.0)
    return {'query': query, 'Precision@5': precision_at_k, 'Recall@5': recall_at_k, 'MRR': reciprocal_rank, 'retrieved': retrieved, 'hits': hits}

if RELEVANT:
    evaluation = [evaluate_query(query, relevant) for query, relevant in RELEVANT.items()]
    for result in evaluation:
        print(result)
    print('Mean Precision@5:', sum(item['Precision@5'] for item in evaluation) / len(evaluation))
    print('Mean Recall@5:', sum(item['Recall@5'] for item in evaluation) / len(evaluation))
    print('Mean MRR:', sum(item['MRR'] for item in evaluation) / len(evaluation))
else:
    print('RELEVANT is empty. Hãy tự gắn labels trước khi tính metrics.')

In [ ]:
# Chọn ít nhất một query tốt và một query kém sau khi xem kết quả.
ERROR_ANALYSIS_QUERIES = []
for query in ERROR_ANALYSIS_QUERIES:
    print('\nERROR CASE:', query)
    for rank, document_id, score in search(query, models['B_normalized'], normalized, top_k=5):
        print(rank, document_id, round(score, 6), texts[document_id][:500].replace('\n', ' '))